# Data Processing

##### Import Libraries

In [ ]:
import os
import sys
import h5py
import torch
import numpy as np
import torch.nn.functional as F
import torchvision.transforms as T
import torchaudio.transforms as transforms
from torch.utils.data import Dataset, DataLoader


curr_dir = os.getcwd()
root_dir = os.path.dirname(curr_dir)
data_dir = os.path.join(root_dir, 'data')

if root_dir not in sys.path:
	sys.path.append(root_dir)

import aceverify

/rhome/abhar061/bigdata/.conda/envs/aceverify/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### Resolving Dependencies (Run only in HPCC)

In [ ]:
# Loading ffmpeg in HPCC
ffmpeg_lib_path = "/opt/linux/rocky/8.x/x86_64/pkgs/ffmpeg/5.0/lib"

if "LD_LIBRARY_PATH" in os.environ:
    os.environ["LD_LIBRARY_PATH"] += os.pathsep + ffmpeg_lib_path
else:
    os.environ["LD_LIBRARY_PATH"] = ffmpeg_lib_path

print(f"LD_LIBRARY_PATH updated to include: {ffmpeg_lib_path}")

ffmpeg_dir = "/opt/linux/rocky/8.x/x86_64/pkgs/ffmpeg/5.0/bin"
ffmpeg_exe = "/opt/linux/rocky/8.x/x86_64/pkgs/ffmpeg/5.0/bin/ffmpeg"

os.environ["PATH"] += os.pathsep + ffmpeg_dir
os.environ["FFMPEG_BIN"] = ffmpeg_exe

print(f"PATH updated. FFMPEG_BIN set to: {os.environ['FFMPEG_BIN']}")

##### Define data specific constraints

In [2]:
zip_file = 'dfdc_train_part_48.zip'
zip_file_path = os.path.join(data_dir, zip_file)
subfolder = 'dfdc_train_part_0'
h5_path = 'processed_data.h5'

##### Preprocess Audio & Video

In [ ]:
aceverify.preprocess_dataset(zip_file_path, subfolder, h5_path)

##### Create TorchScript Checkpoint for CPU efficient inference

In [ ]:
checkpoint_dir = os.path.join(data_dir, 'trained_model_paths')

# Load the regular checkpoint (.pth/.pt) of the trained model
model = aceverify.ACEVerifyModel()
model.load_state_dict(torch.load(os.path.join(checkpoint_dir, "aceverify_model.pth"), map_location='cpu'))
model.eval()

# Create dummy input (B, C, T, H, W) -> (1, 3, 32, 224, 224)
dummy_input = torch.randn(1, 3, 32, 224, 224)

# Trace the model
traced_model = torch.jit.trace(model, dummy_input)

# Save optimized version
traced_model.save(os.path.join(checkpoint_dir, "aceverify_optimized.pt"))

##### 